# Elección del modelo: Qwen 2.5 3B-Instruct

**¿Por qué Qwen 2.5 3B?**
- Ya está en producción en NormaBot via Ollama para la etapa de grading
- Open-weight, totalmente compatible con LoRA y QLoRA
- Suficientemente ligero para entrenarlo en una GPU con ≥ 8 GB VRAM
- Muy buena capacidad semántica en español para su tamaño
- La tarea es clasificación binaria simple, no requiere un modelo más grande

**Parámetros:** ~3.000 millones  
**Cuantización durante entrenamiento:** 4-bit NF4 (QLoRA, `bitsandbytes`)  
**Adaptadores LoRA:** r=8, alpha=16, capas de atención (`q_proj`, `k_proj`, `v_proj`, `o_proj`)

> Entrenado localmente en **NVIDIA RTX 4070 Super (13 GB VRAM)**.  
> Compatible con cualquier GPU ≥ 8 GB VRAM (RTX 3080, A10G, etc.).

# Instalación de librerías y comprobación de entorno


In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\rammu\anaconda3\envs\venv_finetuning\python.exe
3.12.10 | packaged by conda-forge | (main, Apr 10 2025, 22:08:16) [MSC v.1943 64 bit (AMD64)]


In [2]:
# Dependencias necesarias — instalar antes de ejecutar este notebook:
#
#   GPU NVIDIA (CUDA 12.4 — RTX 4070 Super / driver >= 525):
#     pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#
#   Resto del stack:
#     pip install -r requirements/finetuning.txt

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" | GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    print("\nSin GPU: el entrenamiento será muy lento. Se recomienda GPU con >= 8 GB VRAM.")

PyTorch 2.6.0+cu124 | CUDA: True | GPU: NVIDIA GeForce RTX 4070 SUPER (13 GB)


In [3]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# Asegura que el directorio de trabajo sea la raíz del repositorio
# (necesario si Jupyter se abrió desde src/finetuning/ en vez de desde la raíz)
_here = Path.cwd()
if not (_here / "requirements").exists():
    for _parent in _here.parents:
        if (_parent / "requirements").exists():
            os.chdir(_parent)
            break
    else:
        raise RuntimeError(
            f"No se encontró la raíz del repositorio desde {_here}.\n"
            "Ejecuta Jupyter desde la raíz del proyecto: jupyter lab"
        )

# Importar funciones auxiliares del módulo local
sys.path.insert(0, str(Path.cwd() / "src" / "finetuning"))
from functions import (
    LABEL_RELEVANTE, LABEL_NO_RELEVANTE, LABELS, GRADING_SYSTEM_PROMPT,
    check_gpu, load_grading_dataset, split_dataset, show_split_stats,
    build_grading_messages, format_training_prompt, examples_to_hf_dataset,
    load_tokenizer, load_model_4bit, build_peft_model, get_training_args,
    run_training, save_adapter, predict_relevance, evaluate_model,
    print_comparison, log_experiment,
)

check_gpu()

PyTorch:          2.6.0+cu124
CUDA disponible:  True
GPU:              NVIDIA GeForce RTX 4070 SUPER
VRAM total:       12.9 GB


In [4]:
from pathlib import Path

# El CWD ya fue fijado a la raíz del repositorio en la celda anterior (env-check-05)

# Rutas locales (relativas a la raíz del repositorio)
DATASET_PATH = Path("data/processed/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_DIR   = "src/finetuning/output/qwen-grader-lora"
ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"
MERGED_PATH  = "src/finetuning/output/qwen-grader-merged"

MAX_SEQ_LEN = 512

print(f"Directorio de trabajo: {Path.cwd()}")
print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")

Directorio de trabajo: c:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final
Configuración:
  Modelo base:    Qwen/Qwen2.5-3B-Instruct
  Dataset:        data\processed\grading_dataset.jsonl  (existe: True)
  Output dir:     src/finetuning/output/qwen-grader-lora
  Adapter path:   src/finetuning/output/qwen-grader-lora/adapter_final
  Max seq len:    512


In [5]:
all_data = load_grading_dataset(DATASET_PATH)

Dataset cargado: C:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final\data\processed\grading_dataset.jsonl
  Total ejemplos:  634
  Relevantes:      283 (44.6%)
  No relevantes:   351 (55.4%)

Muestra:
{
  "query": "¿Cuándo es obligatorio recurrir a un organismo notificado para evaluar conformidad de IA?",
  "document": "Artículo 1258 del Código Civil — Contratos. Los contratos se perfeccionan por el mero consentimiento, y desde entonces obligan no sólo al cumplimiento de lo expresamente pactado, sino también a todas las consecuencias que, según su naturaleza, sean conformes a la buena fe, al uso y a la ley.",
  "label": "no relevante"
}


In [6]:
train_data, val_data, test_data = split_dataset(all_data)

Split del dataset:
  Train: 443 ejemplos
  Val:   95 ejemplos
  Test:  96 ejemplos


In [7]:
tokenizer  = load_tokenizer(MODEL_ID)
base_model = load_model_4bit(MODEL_ID)

c:\Users\rammu\anaconda3\envs\venv_finetuning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando tokenizador...
Tokenizador listo. Vocabulario: 151,643 tokens
Cargando Qwen/Qwen2.5-3B-Instruct en 4-bit NF4 (evaluacion (4-bit))...


Loading weights: 100%|██████████| 434/434 [00:03<00:00, 123.79it/s, Materializing param=model.norm.weight]                              


Modelo cargado. Dispositivo: cuda:0


# Formato del prompt de entrenamiento

Para el fine-tuning usamos el **chat template nativo de Qwen 2.5** (formato `<|im_start|>`).
Cada ejemplo de entrenamiento tiene la forma:

```
<|im_start|>system
Eres un asistente especializado en normativa de IA...<|im_end|>
<|im_start|>user
Consulta: {query}

Documento: {document}

¿Es este documento relevante para responder la consulta?<|im_end|>
<|im_start|>assistant
{label}<|im_end|>
```

El modelo aprende a generar directamente `relevante` o `no relevante` como turno de assistant.


In [8]:
sample_formatted = format_training_prompt(train_data[0], tokenizer)
print("Ejemplo de prompt de entrenamiento:")
print("-" * 65)
print(sample_formatted)
print("-" * 65)
print(f"Longitud: {len(sample_formatted)} chars")

Ejemplo de prompt de entrenamiento:
-----------------------------------------------------------------
<|im_start|>system
Eres un asistente especializado en normativa de inteligencia artificial. Tu tarea es determinar si un documento contiene información útil para responder una consulta sobre regulación de IA (EU AI Act, BOE, normativa española). Responde únicamente con 'relevante' o 'no relevante', sin explicación adicional.<|im_end|>
<|im_start|>user
Consulta: ¿El EU AI Act tiene su propio régimen de sanciones o depende del RGPD?

Documento: Artículo 83 RGPD — Condiciones generales para la imposición de multas administrativas. Las infracciones de los principios básicos del tratamiento y los derechos de los interesados se sancionarán con multas de hasta 20.000.000 EUR o el 4 % del volumen de negocio anual global. Otras infracciones (obligaciones del responsable, del encargado, organismos de certificación) se sancionarán con multas de hasta 10.000.000 EUR o el 2 % del volumen de negocio

In [9]:
train_dataset = examples_to_hf_dataset(train_data, tokenizer)
val_dataset   = examples_to_hf_dataset(val_data,   tokenizer)

print(f"Dataset de entrenamiento: {len(train_dataset)} ejemplos")
print(f"Dataset de validacion:    {len(val_dataset)} ejemplos")

Dataset de entrenamiento: 443 ejemplos
Dataset de validacion:    95 ejemplos


# Fine-tuning con QLoRA

**QLoRA** (Quantized Low-Rank Adaptation) combina:
- **Cuantización 4-bit NF4** para reducir el uso de VRAM
- **LoRA** para entrenar solo un subconjunto pequeño de parámetros

Con r=8 entrenamos aproximadamente **el 1-2% de los parámetros totales** del modelo,
lo que hace el fine-tuning viable en una GPU T4 (16 GB VRAM).

Usamos `SFTTrainer` de `trl` (Supervised Fine-Tuning Trainer), que gestiona automáticamente
el packing de secuencias, la máscara de padding y la pérdida solo sobre el turno de assistant.


In [10]:
try:
    del base_model
    torch.cuda.empty_cache()
    print("Modelo baseline liberado de VRAM.")
except NameError:
    pass

model = load_model_4bit(MODEL_ID, for_training=True)

Modelo baseline liberado de VRAM.
Cargando Qwen/Qwen2.5-3B-Instruct en 4-bit NF4 (entrenamiento (QLoRA))...


Loading weights: 100%|██████████| 434/434 [00:03<00:00, 126.40it/s, Materializing param=model.norm.weight]                              


Modelo cargado. Dispositivo: cuda:0


In [11]:
model, lora_config = build_peft_model(model)

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827


In [12]:
training_args = get_training_args(OUTPUT_DIR)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments configurados:
  Epochs:         3
  Batch efectivo: 4 x 4 = 16
  Learning rate:  0.0002
  LR scheduler:   SchedulerType.COSINE
  Optimizador:    OptimizerNames.PAGED_ADAMW_8BIT


In [13]:
train_result = run_training(
    model, training_args, train_dataset, val_dataset, tokenizer, MAX_SEQ_LEN
)

Map: 100%|██████████| 95/95 [00:00<00:00, 8383.67 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Iniciando entrenamiento QLoRA... (81 pasos estimados)


c:\Users\rammu\anaconda3\envs\venv_finetuning\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,1.183347,0.730890
2,0.437550,0.385197
3,0.267772,0.330453


c:\Users\rammu\anaconda3\envs\venv_finetuning\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\rammu\anaconda3\envs\venv_finetuning\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



Entrenamiento completado.
  Loss final:    0.7306
  Tiempo total:  1435.0s


# Guardado del modelo

Guardamos el **adaptador LoRA** en `src/finetuning/output/qwen-grader-lora/adapter_final/`.
El adaptador pesa solo unos pocos MB y es suficiente para inferencia si se carga junto al modelo base.

Para producción con Ollama, descomenta el bloque de merge para obtener el modelo completo
y convertirlo a GGUF.

In [14]:
metadata = {
    "model_id":              MODEL_ID,
    "task":                  "rag_relevance_grading",
    "labels":                LABELS,
    "lora_r":                lora_config.r,
    "lora_alpha":            lora_config.lora_alpha,
    "train_size":            len(train_data),
    "val_size":              len(val_data),
    "test_size":             len(test_data),
    "grading_system_prompt": GRADING_SYSTEM_PROMPT,
}
save_adapter(model, tokenizer, ADAPTER_PATH, metadata)

# OPCIONAL: merge adaptador + modelo base para convertir a GGUF
# os.makedirs(MERGED_PATH, exist_ok=True)
# from peft import PeftModel
# base_for_merge = load_model_4bit(MODEL_ID)
# merged = PeftModel.from_pretrained(base_for_merge, ADAPTER_PATH)
# merged = merged.merge_and_unload()
# save_adapter(merged, tokenizer, MERGED_PATH, metadata)
# print("Siguiente paso: convertir a GGUF con llama.cpp y subir a Ollama.")

Adaptador LoRA guardado en: src/finetuning/output/qwen-grader-lora/adapter_final
Metadatos guardados en: src/finetuning/output/qwen-grader-lora/adapter_final/model_metadata.json
